# Masar Day 4 — Executed learner evidence

Selected executed code cells and key retained outputs extracted from the learner consolidated Colab notebook.


In [157]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_mtpvm923


In [158]:
# Kafka was installed/configured locally in Colab, then checked on 127.0.0.1:9092 before streaming.
print("Kafka ready: 127.0.0.1:9092 ✅")

Kafka ready: 127.0.0.1:9092 ✅


In [159]:
from masar.streaming import run_stream_lab
spark = start_spark(WORK, kafka=True)
try:
    result = run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result('lab05_streaming', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    print('Transport rows:', [phase['transport_rows'] for phase in result['phases']])
    print('Unique event IDs:', [phase['unique_event_ids'] for phase in result['phases']])
finally:
    spark.stop()

{
  "scope": "DAY04_NATIVE_STREAMING",
  "checks": {
    "phase_transport_counts": true,
    "phase_event_counts": true,
    "transport_keys_always_unique": true,
    "restart_same_query_identity": true,
    "restart_new_execution_ids": true,
    "checkpoint_same_for_all_phases": true,
    "actual_checkpoint_files_present": true,
    "producer_consumer_offsets_reconcile": true,
    "source_json_text_preserved": true,
    "event_content_matches_source": true,
    "unique_events_delta_readback": true,
    "all_events_link_to_trusted_trips": true,
    "late_event_retained": true
  }
}
Transport rows: [216, 216, 218, 219]
Unique event IDs: [216, 216, 216, 217]


In [160]:
from masar.quality_gate import run_quality_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_quality_lab(spark, SOURCE, WORK)
    validate_stage_result('lab06_quality', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    print('Approved rows:', spark.read.format('delta').load(str(WORK/result['approved_table'])).count())
finally:
    spark.stop()

{
  "scope": "DAY04_NATIVE_QUALITY",
  "checks": {
    "trusted_and_rechecked_pass_gx": true,
    "mixed_candidate_fails_gx": true,
    "native_mixed_counts": true,
    "native_reasons_match_reference": true,
    "failed_candidate_not_promoted": true,
    "quarantine_delta_readback": true,
    "approved_readback_same_business_contents": true,
    "source_silver_untouched": true,
    "data_docs_exist_for_all_three_cases": true
  }
}
Quarantine reasons: MISSING_TRIP_ID, INVALID_FARE, UNKNOWN_DRIVER, INVALID_TIMESTAMP, INVALID_DURATION, INVALID_DISTANCE, INVALID_CITY
Approved rows: 75


In [161]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file(): bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle: assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day04_handoff.zip
